        # 전체 Colab 파이프라인 — All in One

        기존 `notebook/`의 8개 실행 노트북을 한 파일에 합친 마스터 노트북이다.
        반복되던 저장소 clone·Google Drive·dependency 셀은 아래 공통 런타임으로 한 번만
        통합했고, 각 노트북의 고유 셀은 원래 실행 순서대로 모두 포함했다.

        > **권장 사용법:** 공통 런타임을 먼저 실행한 뒤 필요한 PART의 `RUN_*` 스위치만 켠다.
        > 기본값에서는 비용이 큰 학습·추론·artifact 생성과 테스트가 실행되지 않는다.
        > `90`은 validation label 학습 사용이 명시적으로 승인된 경우에만 연다.

        ## 목차

        1. [환경·데이터 준비](#part-00)
2. [Zero-shot 기준선](#part-01)
3. [CPU 기준선](#part-02)
4. [Qwen scorer 훈련](#part-03)
5. [교정·앙상블·추론](#part-04)
6. [Rationale·최종화](#part-05)
7. [제출·테스트](#part-06)
8. [선택적 최종 재훈련](#part-90)


<a id="common-runtime"></a>

# 공통 런타임 — 저장소, Drive, 의존성

한 세션에서 한 번만 실행한다. 마스터 노트북은 세션 종료 후 artifact와 Hugging Face
cache를 보존하기 위해 `USE_GOOGLE_DRIVE=True`를 기본값으로 사용한다.


## 공통 저장소와 영속 경로


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = True
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


## 공통 전체 파이프라인 의존성 설치


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]", "tqdm>=4.66", "sentencepiece>=0.2"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


---

<a id="part-00"></a>

# PART 1 · 환경·데이터 준비

**원본 노트북:** `00_colab_environment_and_data.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 00. Colab 환경 및 데이터 준비
저장소를 고정된 위치에 준비하고, 의존성·데이터·테스트·fold를 검증한다.

- 입력: 추적된 `dataset/*.jsonl`, 기존 zero-shot 결과
- 출력: `artifacts/reports/*`, `artifacts/folds/*`
- 다음 단계: `01_zero_shot_baseline.ipynb` 또는 `02_cpu_baselines.ipynb`

긴 학습을 이어 할 경우 모든 세션에서 같은 `PROJECT_ROOT`, `artifacts` Drive 경로,
`HF_HOME`을 사용해야 한다. 이미 생성된 artifact를 다른 경로로 옮기지 않는다.


### 3. 입력 및 런타임 사전 점검

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
ZERO_SHOT_PATH = (
    PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"
    / "qwen3_14b_zero_shot_strict_v2_validation.jsonl"
)
require_paths(TRAIN_PATH, VALIDATION_PATH, ZERO_SHOT_PATH)

def count_jsonl(path: Path) -> int:
    with path.open("r", encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())

total, used, free = shutil.disk_usage(PROJECT_ROOT)
print({
    "python": sys.version.split()[0],
    "train_rows": count_jsonl(TRAIN_PATH),
    "validation_rows": count_jsonl(VALIDATION_PATH),
    "disk_free_gb": round(free / 1024**3, 1),
})
assert (3, 10) <= sys.version_info[:2] < (3, 13)


### 4. 실행 스위치
테스트만 기본 실행한다. 감사와 fold CLI는 산출물을 덮어쓰지 않으므로, 처음 실행할
때만 스위치를 켠다. 이미 결과가 있으면 삭제하지 말고 그대로 재사용한다.


In [ ]:
RUN_UNIT_TESTS = False
RUN_DATA_AUDIT = False
RUN_ZERO_SHOT_REAUDIT = False
RUN_MAIN_FOLDS = False
RUN_LOPO_FOLDS = False


In [ ]:
run_cli("unit tests", RUN_UNIT_TESTS, "-m", "pytest", "-q")


### 5. 데이터 및 기존 zero-shot 결과 감사

In [ ]:
run_cli(
    "data audit",
    RUN_DATA_AUDIT,
    "scripts/audit_data.py",
    "--config", "configs/data.yaml",
)
run_cli(
    "zero-shot re-audit",
    RUN_ZERO_SHOT_REAUDIT,
    "scripts/reproduce_zero_shot.py",
    "--config", "configs/data.yaml",
)

show_json(PROJECT_ROOT / "artifacts" / "reports" / "data_audit.json")
show_json(PROJECT_ROOT / "artifacts" / "reports" / "zero_shot_reaudit.json")


### 6. 고정 5-fold 및 LOPO 진단 fold

In [ ]:
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
LOPO_PATH = PROJECT_ROOT / "artifacts" / "folds" / "lopo_by_prompt.jsonl"

run_cli(
    "main 5-fold",
    RUN_MAIN_FOLDS,
    "scripts/make_folds.py",
    "--config", "configs/data.yaml",
    "--n-folds", "5",
    "--seed", "42",
    "--output", FOLDS_PATH,
)
run_cli(
    "LOPO diagnostic folds",
    RUN_LOPO_FOLDS,
    "scripts/make_lopo_folds.py",
    "--config", "configs/data.yaml",
    "--output", LOPO_PATH,
)

for path in (FOLDS_PATH, LOPO_PATH):
    print(path, "ready" if path.exists() else "not created")


### 7. 다음 단계
- 신규 zero-shot 생성이 필요하면 `01_zero_shot_baseline.ipynb`
- CPU OOF 기준선은 `02_cpu_baselines.ipynb`
- validation label을 합친 최종 재훈련은 일반 개발 흐름과 분리된
  `90_optional_final_retraining.ipynb`에서만 수행한다.


---

<a id="part-01"></a>

# PART 2 · Zero-shot 기준선

**원본 노트북:** `01_zero_shot_baseline.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 01. Qwen3-14B Zero-shot 기준선
기존 독립 zero-shot Colab 코드를 출력 없이 정리한 변환본이다. 공식 훈련 파이프라인과
분리된 기준선이며, `RUN_ZERO_SHOT=True`일 때만 모델을 적재하고 생성한다.

- BF16 지원 L4/A100급 GPU 권장
- `MODEL_REVISION`은 40자리 commit SHA로 고정
- 신규 결과는 `artifacts/zero_shot/<RUN_TAG>/`에 기록


In [ ]:
RUN_ZERO_SHOT = False
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False

require_model_revision(MODEL_REVISION, enabled=RUN_ZERO_SHOT)
require_bf16_gpu(enabled=RUN_ZERO_SHOT)


### 2. 설정
`PROJECT_ROOT`는 `데이터셋` 폴더를 포함하는 프로젝트 루트다. Colab Drive를 쓰는 경우 Drive를 마운트한 뒤 경로 후보에 자동으로 잡히는지 확인한다.

전체 점수를 보려면 `MAX_TRAIN_ROWS=None`, `MAX_VALIDATION_ROWS=None`을 유지한다. 먼저 동작 확인만 하려면 각각 5~20 정도로 줄인다.


In [ ]:
import math
import random
import time
import hashlib

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-14B"
RUN_TAG_BASE = "qwen3_14b_zero_shot_colab_v1"
PROMPT_CONTRACT_VERSION = "zero_shot_strict_v2"
MAX_TRAIN_ROWS = None
MAX_VALIDATION_ROWS = None
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 768
GENERATION_BATCH_SIZE = 1
DOMAINS = ["content", "organization", "expression"]

DATA_DIR = PROJECT_ROOT / "dataset"
TRAIN_PATH = DATA_DIR / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = DATA_DIR / "글쓰기채점능력평가2026_validation.jsonl"
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "zero_shot"
require_paths(TRAIN_PATH, VALIDATION_PATH)
print({"train": str(TRAIN_PATH), "validation": str(VALIDATION_PATH), "output_root": str(OUTPUT_ROOT)})


### 3. 데이터 로드
데이터는 JSONL이며, 각 행은 `prompt`, `essay`, `score`를 포함한다. `score`에는 세 영역과 평균 점수가 들어 있다.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for domain in DOMAINS:
        df[f"gold_{domain}"] = df["score"].apply(lambda x: float(x[domain]))
    df["gold_average"] = df["score"].apply(lambda x: float(x.get("average", np.mean([x[d] for d in DOMAINS]))))
    return df

train_df = to_dataframe(read_jsonl(TRAIN_PATH))
validation_df = to_dataframe(read_jsonl(VALIDATION_PATH))

print("train", train_df.shape)
print("validation", validation_df.shape)
display(train_df[["id", "prompt_num", "gold_content", "gold_organization", "gold_expression", "gold_average"]].head())


In [ ]:
def score_distribution(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []
    for col in ["gold_content", "gold_organization", "gold_expression", "gold_average"]:
        rows.append({
            "split": name,
            "target": col,
            "count": len(df),
            "mean": df[col].mean(),
            "std": df[col].std(),
            "min": df[col].min(),
            "max": df[col].max(),
        })
    return pd.DataFrame(rows)

display(pd.concat([
    score_distribution(train_df, "train"),
    score_distribution(validation_df, "validation"),
], ignore_index=True).round(4))


### 4. 모델 로드
이 기준선은 학습, calibration, 규칙 기반 fallback을 하지 않는다. 모델이 생성한 텍스트에서 JSON만 파싱하고, 파싱 실패는 실패로 기록한다.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = None
model = None
if RUN_ZERO_SHOT:
    compute_dtype = torch.bfloat16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=False,
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=compute_dtype,
        trust_remote_code=False,
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
    )
    model.eval()
    print("loaded", MODEL_ID, MODEL_REVISION)
else:
    print("RUN_ZERO_SHOT=False: 모델 적재를 건너뜁니다.")


### 5. 프롬프트와 파서
프롬프트는 제출 형식에 맞춰 JSON만 요구한다. 점수는 1~5 범위의 숫자로 받는다. 여기서는 점수 보정이나 fallback을 하지 않으므로, 구조가 깨지면 그대로 실패로 계산한다.


In [ ]:
SYSTEM_PROMPT = '''너는 한국어 논증적 글을 일관되게 직접 채점하는 평가자이다.
essay를 읽고 content, organization, expression 세 기준을 모두 평가하라.
반드시 JSON 객체만 출력하고, Markdown 코드블록이나 추가 설명은 출력하지 마라.
생각 과정은 출력하지 마라.
5점은 매우 우수한 글에만 부여하고, 일반적으로 잘 쓴 글은 3~4점으로 평가하라.
근거가 일반적이거나 구체성이 부족하면 content를 낮추고, 구조는 있으나 연결과 전개가 약하면 organization을 낮추며, 문법·어휘·문장 흐름 문제가 보이면 expression을 낮춰라.
관대하게 점수를 올리지 말고, 감점 사유를 먼저 확인한 뒤 종합 판단하라.
각 score는 1 이상 5 이하의 숫자여야 한다.
rationale은 실제 essay 내용에 근거해 한국어 1~2문장으로 작성하라.
'''.strip()


def build_user_prompt(row: pd.Series) -> str:
    return f'''[평가 기준]
content: 주제 이해, 입장 명확성, 주장과 근거의 타당성, 논증의 깊이
organization: 서론-본론-결론 구조, 문단 구성, 전개 순서, 연결성
expression: 문장 정확성, 어휘 선택, 문법, 표현의 자연스러움

[점수 앵커]
5: 결함이 거의 없고, 주장·근거·구성·표현이 모두 매우 우수함
4: 대체로 우수하지만 일부 근거의 구체성, 연결, 표현상 아쉬움이 있음
3: 기본 요건은 충족하지만 근거가 일반적이거나 전개·표현의 약점이 뚜렷함
2: 입장, 근거, 구성, 표현 중 여러 요소가 부족해 설득력이 낮음
1: 주제 이해, 논증 구조, 문장 표현이 전반적으로 매우 미흡함

[출력 형식]
{{
  "content": {{"score": 1, "rationale": "content 판단 근거"}},
  "organization": {{"score": 1, "rationale": "organization 판단 근거"}},
  "expression": {{"score": 1, "rationale": "expression 판단 근거"}}
}}

[prompt_text]
{row['prompt']}

[essay_text]
{row['essay']}
'''.strip()


def apply_chat_template_no_thinking(messages: list[dict]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def make_prompt(row: pd.Series) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]
    return apply_chat_template_no_thinking(messages)



RUN_CONTRACT = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "prompt_contract_version": PROMPT_CONTRACT_VERSION,
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "seed": SEED,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "do_sample": False,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(RUN_CONTRACT, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()
RUN_TAG = f"{RUN_TAG_BASE}_{RUN_SIGNATURE[:12]}"
OUTPUT_DIR = OUTPUT_ROOT / RUN_TAG
if RUN_ZERO_SHOT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"run_tag": RUN_TAG, "run_signature": RUN_SIGNATURE, "output": str(OUTPUT_DIR)})


In [ ]:
def repair_common_json_errors(text: str) -> str:
    first = text.find("{")
    last = text.rfind("}")
    candidate = text[first:last + 1] if first >= 0 and last > first else text

    # Common Qwen typo: rationale string closes with ")" instead of "}"
    candidate = re.sub(
        r'("rationale"\s*:\s*"[^"\\]*(?:\\.[^"\\]*)*")\s*\)',
        r"\1}",
        candidate,
        flags=re.S,
    )

    # Remove trailing commas before object/list close
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
    return candidate


def extract_first_json_object(text: str):
    decoder = json.JSONDecoder()

    repaired = repair_common_json_errors(text)
    try:
        return decoder.raw_decode(repaired[repaired.find("{"):])[0]
    except Exception:
        pass

    # Fallback: scan every opening brace, but only accept full schema later.
    for i, ch in enumerate(repaired):
        if ch != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(repaired[i:])
            if isinstance(obj, dict) and all(d in obj for d in DOMAINS):
                return obj
        except json.JSONDecodeError:
            continue

    raise ValueError("No valid JSON object found")


def extract_domain_by_regex(text: str, domain: str) -> dict:
    m = re.search(rf'"{domain}"\s*:\s*\{{', text)
    if not m:
        raise ValueError(f"missing object: {domain}")

    body = text[m.end():]
    next_domain = re.search(r',\s*"(content|organization|expression)"\s*:', body)
    if next_domain:
        body = body[:next_domain.start()]

    score_m = re.search(
        r'"score"\s*:\s*(?:"(-?(?:\d+(?:\.\d*)?|\.\d+))"|(-?(?:\d+(?:\.\d*)?|\.\d+)))(?=\s*[,}])',
        body,
    )
    rationale_m = re.search(r'"rationale"\s*:\s*"((?:\\.|[^"\\])*)"', body, flags=re.S)

    if not score_m:
        raise ValueError(f"missing score: {domain}")
    if not rationale_m:
        raise ValueError(f"missing rationale: {domain}")

    score_text = score_m.group(1) or score_m.group(2)
    rationale_raw = rationale_m.group(1)
    try:
        rationale = json.loads(f'"{rationale_raw}"')
    except Exception:
        rationale = rationale_raw.replace('\\"', '"').strip()

    return {
        "score": float(score_text),
        "rationale": rationale,
    }


def validate_prediction_object(obj: dict) -> dict:
    if not isinstance(obj, dict):
        raise ValueError("prediction is not a JSON object")

    clean = {}
    for domain in DOMAINS:
        if domain not in obj or not isinstance(obj[domain], dict):
            raise ValueError(f"missing object: {domain}")

        score = obj[domain].get("score")
        rationale = obj[domain].get("rationale")

        if isinstance(score, str):
            score_text = score.strip()
            if re.fullmatch(r"-?(?:\d+(?:\.\d*)?|\.\d+)", score_text) is None:
                raise ValueError(f"non-numeric score: {domain}")
            score = float(score_text)

        if isinstance(score, bool) or not isinstance(score, (int, float)):
            raise ValueError(f"non-numeric score: {domain}")

        score = float(score)
        if not (1.0 <= score <= 5.0):
            raise ValueError(f"score out of range: {domain}={score}")

        if not isinstance(rationale, str) or not rationale.strip():
            raise ValueError(f"empty rationale: {domain}")

        clean[domain] = {
            "score": score,
            "rationale": rationale.strip(),
        }

    return clean


def parse_model_output(text: str) -> dict:
    try:
        obj = extract_first_json_object(text)
        clean = validate_prediction_object(obj)
        return {"ok": True, "parsed": clean, "error": None}
    except Exception as first_exc:
        try:
            repaired_text = repair_common_json_errors(text)
            clean = {
                domain: extract_domain_by_regex(repaired_text, domain)
                for domain in DOMAINS
            }
            clean = validate_prediction_object(clean)
            return {"ok": True, "parsed": clean, "error": None}
        except Exception as second_exc:
            return {
                "ok": False,
                "parsed": None,
                "error": f"{first_exc}; regex fallback failed: {second_exc}",
            }


def _parser_contract_probe(score_literal: str) -> str:
    return (
        '{"content":{"score":' + score_literal + ',"rationale":"근거"},'
        '"organization":{"score":3,"rationale":"근거"},'
        '"expression":{"score":3,"rationale":"근거"}}'
    )


assert parse_model_output(_parser_contract_probe("3.5"))["ok"]
for invalid_score in ("true", "10", "50", '\"10\"', '\"50\"', '\"1abc\"'):
    assert not parse_model_output(_parser_contract_probe(invalid_score))["ok"]


### 6. 단일 샘플 추론 확인
첫 샘플에서 출력 형식이 안정적인지 확인한다. 이 셀이 실패하면 전체 루프를 돌리지 말고 프롬프트나 `MAX_NEW_TOKENS`를 먼저 조정한다.


In [ ]:
@torch.inference_mode()
def generate_batch(rows: list[pd.Series]) -> list[str]:
    if model is None or tokenizer is None:
        raise RuntimeError("먼저 RUN_ZERO_SHOT=True로 모델을 적재하세요.")
    prompts = [make_prompt(row) for row in rows]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_width = inputs["input_ids"].shape[-1]
    return [
        tokenizer.decode(row[prompt_width:], skip_special_tokens=True).strip()
        for row in output_ids
    ]


def generate_one(row: pd.Series) -> str:
    return generate_batch([row])[0]


if RUN_ZERO_SHOT:
    sample_output = generate_one(validation_df.iloc[0])
    print(sample_output)
    print(parse_model_output(sample_output))
else:
    print("샘플 생성 생략")


### 7. 전체/부분 실행 루프
각 레코드는 생성 직후 JSONL로 저장된다. Colab 세션이 끊기면 같은 셀을 다시 실행해 이미 저장된 `id`를 건너뛸 수 있다. GPU 여유가 있으면 `GENERATION_BATCH_SIZE`를 3 또는 4로 올려 한 번에 여러 글을 생성한다.


In [ ]:
def select_rows(df: pd.DataFrame, limit: int | None) -> pd.DataFrame:
    if limit is None:
        return df.copy()
    return df.head(limit).copy()


def prediction_path(split_name: str) -> Path:
    return OUTPUT_DIR / f"{RUN_TAG}_{split_name}.jsonl"


def load_prediction_records(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def iter_row_batches(df: pd.DataFrame, batch_size: int):
    for start in range(0, len(df), batch_size):
        yield [row for _, row in df.iloc[start:start + batch_size].iterrows()]


def run_split(df: pd.DataFrame, split_name: str) -> list[dict]:
    path = prediction_path(split_name)
    existing = load_prediction_records(path)
    mismatched = [
        row.get("id") for row in existing
        if row.get("run_signature") != RUN_SIGNATURE
    ]
    if mismatched:
        raise RuntimeError(
            f"output에 다른 revision/생성 계약의 행이 섞여 있습니다: {mismatched[:5]}"
        )
    done_ids = {
        row["id"] for row in existing
        if row.get("run_signature") == RUN_SIGNATURE
    }
    rows = df[~df["id"].isin(done_ids)].copy()
    print({
        "split": split_name,
        "target_rows": len(df),
        "already_done": len(done_ids),
        "remaining": len(rows),
        "batch_size": GENERATION_BATCH_SIZE,
        "path": str(path),
    })

    with path.open("a", encoding="utf-8") as f:
        for batch_rows in tqdm(
            iter_row_batches(rows, GENERATION_BATCH_SIZE),
            total=math.ceil(len(rows) / GENERATION_BATCH_SIZE) if len(rows) else 0,
            desc=split_name,
        ):
            started = time.time()
            raw_outputs = generate_batch(batch_rows)
            elapsed = time.time() - started
            per_item_elapsed = elapsed / max(len(batch_rows), 1)

            for row, raw_output in zip(batch_rows, raw_outputs):
                parsed = parse_model_output(raw_output)
                record = {
                    "run_tag": RUN_TAG,
                    "run_signature": RUN_SIGNATURE,
                    "run_contract": RUN_CONTRACT,
                    "model_id": MODEL_ID,
                    "model_revision": MODEL_REVISION,
                    "split": split_name,
                    "id": row["id"],
                    "prompt_num": row.get("prompt_num"),
                    "gold_score": row["score"],
                    "raw_output": raw_output,
                    "parse_ok": parsed["ok"],
                    "parsed": parsed["parsed"],
                    "parse_error": parsed["error"],
                    "elapsed_sec": round(per_item_elapsed, 3),
                    "batch_size": len(batch_rows),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

    return load_prediction_records(path)

train_eval_df = select_rows(train_df, MAX_TRAIN_ROWS)
validation_eval_df = select_rows(validation_df, MAX_VALIDATION_ROWS)

print("train_eval", train_eval_df.shape)
print("validation_eval", validation_eval_df.shape)


In [ ]:
# validation을 먼저 실행한다. append-resume는 같은 RUN_TAG의 완료 ID를 건너뛴다.
validation_records = (
    run_split(validation_eval_df, "validation")
    if RUN_ZERO_SHOT
    else load_prediction_records(prediction_path("validation"))
)
print("validation records", len(validation_records))


In [ ]:
RUN_TRAIN_DRIFT_CHECK = False
train_records = (
    run_split(train_eval_df, "train")
    if RUN_ZERO_SHOT and RUN_TRAIN_DRIFT_CHECK
    else load_prediction_records(prediction_path("train"))
)
print("train records", len(train_records))


### 8. 평가 지표 계산
이 평가는 raw zero-shot 기준선이다. calibration, threshold 조정, fallback 점수 대입을 하지 않는다. 따라서 파싱 실패율도 점수와 함께 봐야 한다.


In [ ]:
def records_to_pred_df(records: list[dict]) -> pd.DataFrame:
    rows = []
    for r in records:
        row = {
            "id": r["id"],
            "parse_ok": bool(r.get("parse_ok")),
            "parse_error": r.get("parse_error"),
            "elapsed_sec": r.get("elapsed_sec"),
        }
        parsed = r.get("parsed") if r.get("parse_ok") else None
        if parsed:
            for domain in DOMAINS:
                row[f"pred_{domain}"] = float(parsed[domain]["score"])
            row["pred_average"] = float(np.mean([row[f"pred_{d}"] for d in DOMAINS]))
        rows.append(row)
    pred_df = pd.DataFrame(rows)
    if not pred_df.empty:
        pred_df = pred_df.drop_duplicates("id", keep="last")
    return pred_df


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def summarize_metrics(gold_df: pd.DataFrame, records: list[dict], split_name: str):
    pred_df = records_to_pred_df(records)
    merged = gold_df.merge(pred_df, on="id", how="left")
    expected = len(gold_df)
    generated = int(merged["parse_ok"].notna().sum()) if "parse_ok" in merged else 0
    parsed = merged[merged["parse_ok"] == True].copy()

    metric_rows = []
    for target in DOMAINS + ["average"]:
        y_true_col = f"gold_{target}"
        y_pred_col = f"pred_{target}"
        if len(parsed) == 0 or y_pred_col not in parsed:
            metric_rows.append({"split": split_name, "target": target, "rmse": np.nan, "spearman": np.nan, "n": 0})
            continue
        y_true = parsed[y_true_col].astype(float)
        y_pred = parsed[y_pred_col].astype(float)
        corr = spearmanr(y_true, y_pred).correlation if y_pred.nunique() > 1 else np.nan
        metric_rows.append({
            "split": split_name,
            "target": target,
            "rmse": rmse(y_true, y_pred),
            "spearman": float(corr) if corr is not None else np.nan,
            "n": len(parsed),
        })

    coverage = {
        "split": split_name,
        "expected_rows": expected,
        "generated_rows": generated,
        "parsed_rows": len(parsed),
        "missing_rows": expected - generated,
        "parse_fail_rows": generated - len(parsed),
        "parse_success_rate": len(parsed) / generated if generated else 0.0,
        "mean_elapsed_sec": float(pd.to_numeric(merged.get("elapsed_sec"), errors="coerce").mean()),
    }
    return pd.DataFrame(metric_rows), coverage, merged

if validation_records:
    validation_metrics, validation_coverage, validation_joined = summarize_metrics(
        validation_eval_df, validation_records, "validation"
    )
    display(pd.DataFrame([validation_coverage]))
    display(validation_metrics.round(4))
else:
    validation_metrics = pd.DataFrame()
    validation_coverage = {"status": "not_run", "expected_rows": len(validation_eval_df)}
    validation_joined = validation_eval_df.copy()
    validation_joined["parse_ok"] = pd.NA
    validation_joined["parse_error"] = pd.NA
    print(validation_coverage)


In [ ]:
if validation_records:
    metrics_out = OUTPUT_DIR / f"{RUN_TAG}_metrics.csv"
    coverage_out = OUTPUT_DIR / f"{RUN_TAG}_coverage.json"
    validation_metrics.to_csv(metrics_out, index=False, encoding="utf-8-sig")
    coverage_out.write_text(
        json.dumps({"validation": validation_coverage}, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(metrics_out, coverage_out)
else:
    print("저장할 실행 결과가 없습니다.")


### 9. 실패 사례와 샘플 확인
파싱 실패가 있으면 먼저 raw output을 보고 프롬프트를 조정한다. 점수 보정이나 fallback은 이 기준선 노트북에서는 적용하지 않는다.


In [ ]:
def show_failures(joined: pd.DataFrame, split_name: str, n: int = 5):
    fail = joined[(joined["parse_ok"].notna()) & (joined["parse_ok"] != True)].copy()
    print(split_name, "failures", len(fail))
    cols = ["id", "parse_error"]
    display(fail[cols].head(n))
    return fail.head(n)

validation_failures = show_failures(validation_joined, "validation")

In [ ]:
def show_prediction_samples(joined: pd.DataFrame, n: int = 5):
    cols = [
        "id", "prompt_num",
        "gold_content", "pred_content",
        "gold_organization", "pred_organization",
        "gold_expression", "pred_expression",
        "gold_average", "pred_average",
    ]
    available = [c for c in cols if c in joined.columns]
    display(joined[joined["parse_ok"] == True][available].head(n))

show_prediction_samples(validation_joined, n=10)


In [ ]:
# Parse failure analysis
fail_df = validation_joined[
    (validation_joined["parse_ok"].notna()) & (validation_joined["parse_ok"] != True)
].copy()

print("failures:", len(fail_df))
display(fail_df["parse_error"].value_counts().head(20))

# raw output 확인
fail_ids = set(fail_df["id"])
fail_records = [r for r in validation_records if r["id"] in fail_ids]

for r in fail_records[:10]:
    print("=" * 80)
    print("id:", r["id"])
    print("error:", r["parse_error"])
    print(r["raw_output"][:2000])

In [ ]:
parsed_df = validation_joined[validation_joined["parse_ok"] == True].copy()
if parsed_df.empty:
    print("표시할 예측이 없습니다.")
else:
    for domain in DOMAINS:
        print("\n", domain)
        display(pd.DataFrame({
            "gold": parsed_df[f"gold_{domain}"].round(2),
            "pred": parsed_df[f"pred_{domain}"].round(2),
        }).describe().round(3))
        display(parsed_df[f"pred_{domain}"].value_counts().sort_index())


---

<a id="part-02"></a>

# PART 3 · CPU 기준선

**원본 노트북:** `02_cpu_baselines.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 02. CPU 기준선 학습 및 평가
고정 fold에서 mean/surface/TF-IDF OOF와 validation 예측을 만들고, nested TF-IDF,
bootstrap 비교 및 OOF calibration 후보를 검증한다.


### 2. 입력, 출력 namespace 및 실행 스위치

In [ ]:
RUN_NAMESPACE = "cpu_v1"
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
BASELINE_DIR = PROJECT_ROOT / "artifacts" / "predictions" / RUN_NAMESPACE
NESTED_DIR = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_nested"
TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"  # 외부에서 제공해야 함

RUN_BASELINES = False
RUN_NESTED_TFIDF = False
RUN_VALIDATION_EVALUATION = False
RUN_BOOTSTRAP_COMPARISON = False
RUN_TFIDF_CALIBRATION = False
RUN_TEST_PREDICTION = False

require_paths(FOLDS_PATH, TRAIN_PATH, VALIDATION_PATH)
print({"baseline_dir": str(BASELINE_DIR), "nested_dir": str(NESTED_DIR)})


### 3. 고정 CPU 기준선 및 nested TF-IDF

In [ ]:
run_cli(
    "fixed CPU baselines",
    RUN_BASELINES,
    "scripts/run_baselines.py",
    "--config", "configs/data.yaml",
    "--baseline-config", "configs/baselines.yaml",
    "--folds", FOLDS_PATH,
    "--output-dir", BASELINE_DIR,
)
run_cli(
    "nested TF-IDF",
    RUN_NESTED_TFIDF,
    "scripts/run_nested_tfidf.py",
    "--config", "configs/tfidf_nested.yaml",
    "--data-config", "configs/data.yaml",
    "--folds", FOLDS_PATH,
    "--output-dir", NESTED_DIR,
    "--model-name", f"nested_tfidf_{RUN_NAMESPACE}",
)
show_json(BASELINE_DIR / "metrics.json")


### 4. validation 평가와 paired bootstrap

In [ ]:
TFIDF_VALIDATION = BASELINE_DIR / "tfidf_ridge_validation.jsonl"
FIXED_TFIDF_OOF = BASELINE_DIR / "tfidf_ridge_oof.jsonl"
NESTED_OOF = NESTED_DIR / "nested_tfidf_oof.jsonl"

run_cli(
    "validation metrics",
    RUN_VALIDATION_EVALUATION,
    "scripts/evaluate_predictions.py",
    "--config", "configs/data.yaml",
    "--gold", VALIDATION_PATH,
    "--pred", TFIDF_VALIDATION,
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_validation.json",
    "--strict",
)
run_cli(
    "nested vs fixed bootstrap",
    RUN_BOOTSTRAP_COMPARISON,
    "scripts/bootstrap_compare.py",
    "--gold", TRAIN_PATH,
    "--candidate", NESTED_OOF,
    "--baseline", FIXED_TFIDF_OOF,
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_nested_vs_fixed.json",
)


### 5. OOF calibration 및 외부 test 적용

In [ ]:
CALIBRATOR = PROJECT_ROOT / "artifacts" / "calibrators" / f"{RUN_NAMESPACE}_tfidf_affine.json"
CALIBRATED_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_tfidf_crossfit_calibrated.jsonl"

run_cli(
    "TF-IDF OOF calibration",
    RUN_TFIDF_CALIBRATION,
    "scripts/fit_calibrator.py",
    "--config", "configs/data.yaml",
    "--gold", TRAIN_PATH,
    "--pred", FIXED_TFIDF_OOF,
    "--folds", FOLDS_PATH,
    "--output", CALIBRATOR,
    "--calibrated-output", CALIBRATED_OOF,
)

if RUN_TEST_PREDICTION:
    require_paths(TEST_PATH, BASELINE_DIR / "tfidf_ridge_ensemble.json")
run_cli(
    "TF-IDF test inference",
    RUN_TEST_PREDICTION,
    "scripts/predict_baseline.py",
    "--input", TEST_PATH,
    "--model", BASELINE_DIR / "tfidf_ridge_ensemble.json",
    "--output", PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_tfidf_test.jsonl",
)


### 6. 승격 판단
validation을 보며 hyperparameter를 고르지 않는다. nested 후보는 train OOF paired
bootstrap에서 고정 TF-IDF보다 낫고 provenance 검사를 통과할 때만 이후 stacker의
`tfidf` source로 사용한다.


---

<a id="part-03"></a>

# PART 4 · Qwen scorer 훈련

**원본 노트북:** `03_qwen_scorer_training.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 03. Qwen3 Scorer QLoRA 훈련
고정 epoch 정책을 먼저 만들고 fold×seed registry를 plan-only로 검토한 다음,
명시적 스위치로 순차 훈련한다. 중단 산출물은 삭제하지 않는다.


### 2. 훈련 파라미터와 fail-fast 검사

In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

EXPERIMENT_ID = f"qwen3_scorer_{RUN_NAMESPACE}"
# ALLOW_MODEL_DOWNLOAD도 registry signature의 일부다. plan 이후 resume까지 바꾸지 않는다.
SEEDS = [42]
SELECTED_FOLDS = [0, 1, 2, 3, 4]
# 이 notebook의 OOF 경로는 완전한 5-fold를 요구한다. fold 집합은 registry 생성 뒤
# immutable이므로 plan/execute 사이에 바꾸지 않는다.
if SELECTED_FOLDS != [0, 1, 2, 3, 4]:
    raise ValueError("정식 OOF 훈련은 SELECTED_FOLDS를 0~4 전체로 유지해야 합니다.")
FIXED_EPOCH = 3

FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
POLICY_PATH = PROJECT_ROOT / "artifacts" / "policies" / f"{EXPERIMENT_ID}_epoch{FIXED_EPOCH}.json"
REGISTRY_PATH = PROJECT_ROOT / "artifacts" / "models" / EXPERIMENT_ID / "registry.json"

CREATE_EPOCH_POLICY = False
PLAN_TRAINING = False
EXECUTE_TRAINING = False
RETRY_PARTIAL = False
VALIDATE_COMPLETE_REGISTRY = False
BUILD_SEED_OOF = False
BUILD_SEED_ENSEMBLE_OOF = False

require_paths(FOLDS_PATH)
require_model_revision(MODEL_REVISION, enabled=PLAN_TRAINING or EXECUTE_TRAINING)
require_bf16_gpu(enabled=EXECUTE_TRAINING)
print({"experiment": EXPERIMENT_ID, "seeds": SEEDS, "folds": SELECTED_FOLDS})


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


### 3. 고정 epoch 정책

In [ ]:
run_cli(
    "fixed epoch policy",
    CREATE_EPOCH_POLICY,
    "scripts/select_fixed_epoch.py",
    "--preselected-epoch", FIXED_EPOCH,
    "--reason", "outer fold 학습 전에 고정한 Colab epoch 정책",
    "--output", POLICY_PATH,
)
show_json(POLICY_PATH)


### 4. fold×seed plan-only registry

In [ ]:
def orchestration_arguments(*, execute: bool) -> list[object]:
    arguments: list[object] = [
        "scripts/orchestrate_scorer_training.py",
        "--experiment-id", EXPERIMENT_ID,
        "--config", "configs/scorer_qlora.yaml",
        "--data-config", "configs/data.yaml",
        "--folds-file", FOLDS_PATH,
        "--epoch-policy", POLICY_PATH,
        "--model-revision", MODEL_REVISION,
    ]
    for seed in SEEDS:
        arguments.extend(["--seed", seed])
    for fold in SELECTED_FOLDS:
        arguments.extend(["--fold", fold])
    if execute:
        arguments.append("--execute")
    if execute and RETRY_PARTIAL:
        arguments.append("--retry-partial")
    arguments.extend(download_flag)
    return arguments

if PLAN_TRAINING:
    require_paths(POLICY_PATH)
run_cli("training plan", PLAN_TRAINING, *orchestration_arguments(execute=False))
show_json(REGISTRY_PATH)


### 5. 실제 훈련
위 registry의 모든 command, fold, seed, fixed epoch, 출력 경로를 확인한 뒤에만
`EXECUTE_TRAINING=True`로 바꾼다. 중단 뒤에도 fold/seed 인자를 바꾸지 않고 동일한
full-registry command를 다시 실행한다. 검증된 완료 task는 건너뛰고 남은 task부터 이어간다.


In [ ]:
if EXECUTE_TRAINING:
    require_paths(POLICY_PATH, REGISTRY_PATH)
run_cli("execute fold training", EXECUTE_TRAINING, *orchestration_arguments(execute=True))


### 6. registry 완결성 검사

In [ ]:
run_cli(
    "validate complete registry",
    VALIDATE_COMPLETE_REGISTRY,
    "scripts/validate_run_registry.py",
    "--registry", REGISTRY_PATH,
    "--require-complete",
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{EXPERIMENT_ID}_registry.json",
)


### 7. fixed-epoch OOF 조립

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"

def checkpoint_dir(seed: int, fold: int) -> Path:
    return (
        PROJECT_ROOT / "artifacts" / "models" / EXPERIMENT_ID
        / f"seed_{seed}__fold_{fold}" / f"epoch_{FIXED_EPOCH}"
    )

seed_oof_paths = {}
for seed in SEEDS:
    output = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_seed{seed}_oof.jsonl"
    seed_oof_paths[seed] = output
    arguments: list[object] = [
        "scripts/build_oof.py", "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
    ]
    for fold in range(5):
        arguments.extend(["--pred", checkpoint_dir(seed, fold) / "oof.jsonl"])
    arguments.extend(["--output", output, "--model", f"{EXPERIMENT_ID}_seed{seed}"])
    run_cli(f"build seed {seed} OOF", BUILD_SEED_OOF, *arguments)

ensemble_oof = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_seed_ensemble_oof.jsonl"
ensemble_arguments: list[object] = [
    "scripts/build_seed_ensemble_oof.py", "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
]
for seed, path in seed_oof_paths.items():
    ensemble_arguments.extend(["--source", f"{seed}={path}"])
ensemble_arguments.extend(["--output", ensemble_oof, "--model", f"{EXPERIMENT_ID}_seed_ensemble"])
if BUILD_SEED_ENSEMBLE_OOF and len(SEEDS) < 2:
    raise ValueError("seed ensemble에는 서로 다른 seed OOF가 최소 2개 필요합니다.")
run_cli("build seed ensemble OOF", BUILD_SEED_ENSEMBLE_OOF, *ensemble_arguments)
print("next OOF:", ensemble_oof if len(SEEDS) > 1 else seed_oof_paths[SEEDS[0]])


### 8. 다음 단계
`04_calibration_ensemble_inference.ipynb`에서 raw OOF provenance를 다시 검증하고
calibration·precision·stacking 후보를 평가한다. outer-held 성능으로 고른 best epoch를
OOF에 사용하지 않는다.


---

<a id="part-04"></a>

# PART 5 · 교정·앙상블·추론

**원본 노트북:** `04_calibration_ensemble_inference.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 04. 교정, 앙상블 및 추론
Qwen raw OOF의 calibration과 precision 비교를 수행하고, Qwen+TF-IDF stacker 및
선택적 anchor/assessment 후보를 만든 뒤 validation/test에 같은 signature로 적용한다.


### 2. 공통 파라미터와 실행 스위치

In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

EXPERIMENT_ID = f"qwen3_scorer_{RUN_NAMESPACE}"
FIXED_EPOCH = 3
SEED = 42
FOLDS = range(5)

TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"  # 외부 입력
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
QWEN_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_seed{SEED}_oof.jsonl"
TFIDF_OOF = PROJECT_ROOT / "artifacts" / "predictions" / "cpu_v1" / "tfidf_ridge_oof.jsonl"
TFIDF_MODEL = PROJECT_ROOT / "artifacts" / "predictions" / "cpu_v1" / "tfidf_ridge_ensemble.json"

RUN_CALIBRATION = False
RUN_BF16_OOF = False
RUN_PRECISION_COMPARISON = False
RUN_VALIDATION_QWEN = False
RUN_TWO_SOURCE_STACKER = False
RUN_VALIDATION_TFIDF = False
RUN_APPLY_STACKER = False
RUN_ANCHOR_REFERENCE_EMBEDDINGS = False
RUN_ANCHOR_OOF = False
RUN_ASSESSMENT_CACHE = False
RUN_ASSESSMENT_FIT = False
RUN_TARGET_QUERY_EMBEDDINGS = False
RUN_TARGET_ANCHOR = False
RUN_TARGET_ASSESSMENT_CACHE = False
RUN_TARGET_ASSESSMENT_PREDICTION = False
RUN_MULTISOURCE_STACKER = False
RUN_APPLY_MULTISOURCE = False
# OOF gate를 통과한 것만 추가한다: [], ["anchor"], ["assessment"], 또는 둘 다.
PROMOTED_AUXILIARY_SOURCES = []

require_model_revision(MODEL_REVISION, enabled=any([
    RUN_BF16_OOF, RUN_VALIDATION_QWEN, RUN_ANCHOR_REFERENCE_EMBEDDINGS, RUN_ASSESSMENT_CACHE,
    RUN_TARGET_QUERY_EMBEDDINGS, RUN_TARGET_ASSESSMENT_CACHE,
]))
require_bf16_gpu(enabled=any([
    RUN_BF16_OOF, RUN_VALIDATION_QWEN, RUN_ANCHOR_REFERENCE_EMBEDDINGS, RUN_ASSESSMENT_CACHE,
    RUN_TARGET_QUERY_EMBEDDINGS, RUN_TARGET_ASSESSMENT_CACHE,
]))


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


In [ ]:
def checkpoint(fold: int) -> Path:
    return (
        PROJECT_ROOT / "artifacts" / "models" / EXPERIMENT_ID
        / f"seed_{SEED}__fold_{fold}" / f"epoch_{FIXED_EPOCH}"
    )

checkpoints = [checkpoint(fold) for fold in FOLDS]
CALIBRATOR = PROJECT_ROOT / "artifacts" / "calibrators" / f"{EXPERIMENT_ID}_affine.json"
CALIBRATED_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_crossfit_calibrated.jsonl"
QWEN_VALIDATION_RAW = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_validation_raw.jsonl"
QWEN_VALIDATION_CALIBRATED = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_validation_calibrated.jsonl"


### 3. OOF calibration

In [ ]:
run_cli(
    "Qwen OOF calibration",
    RUN_CALIBRATION,
    "scripts/fit_calibrator.py",
    "--config", "configs/data.yaml",
    "--gold", TRAIN_PATH,
    "--pred", QWEN_OOF,
    "--folds", FOLDS_PATH,
    "--output", CALIBRATOR,
    "--calibrated-output", CALIBRATED_OOF,
)


### 4. 동일 checkpoint 4bit/BF16 비교

In [ ]:
BF16_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_bf16_oof.jsonl"
precision_arguments: list[object] = [
    "scripts/build_precision_oof.py", "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
]
for path in checkpoints:
    precision_arguments.extend(["--checkpoint", path])
precision_arguments.extend([
    "--precision", "bf16", "--model", EXPERIMENT_ID, "--output", BF16_OOF,
])
precision_arguments.extend(download_flag)
run_cli("build BF16 OOF", RUN_BF16_OOF, *precision_arguments)

run_cli(
    "BF16 vs 4bit",
    RUN_PRECISION_COMPARISON,
    "scripts/compare_precisions.py",
    "--config", "configs/precision_comparison.yaml",
    "--gold", TRAIN_PATH,
    "--folds", FOLDS_PATH,
    "--candidate", BF16_OOF,
    "--baseline", QWEN_OOF,
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{EXPERIMENT_ID}_bf16_vs_4bit.json",
)


### 5. validation Qwen raw/calibrated 추론

In [ ]:
def scorer_arguments(output: Path, *, calibrator: Path | None) -> list[object]:
    arguments: list[object] = ["scripts/predict_scorer.py", "--input", VALIDATION_PATH]
    for path in checkpoints:
        arguments.extend(["--checkpoint", path])
    if calibrator is not None:
        arguments.extend(["--calibrator", calibrator])
    arguments.extend([
        "--precision", "4bit", "--batch-size", "1",
        "--model-name", f"{EXPERIMENT_ID}_seed{SEED}", "--output", output,
    ])
    arguments.extend(download_flag)
    return arguments

run_cli("validation raw Qwen", RUN_VALIDATION_QWEN, *scorer_arguments(QWEN_VALIDATION_RAW, calibrator=None))
run_cli(
    "validation calibrated Qwen",
    RUN_VALIDATION_QWEN,
    *scorer_arguments(QWEN_VALIDATION_CALIBRATED, calibrator=CALIBRATOR),
)


### 6. Qwen + TF-IDF simplex stacker

In [ ]:
STACKER = PROJECT_ROOT / "artifacts" / "stackers" / f"{RUN_NAMESPACE}_qwen_tfidf.json"
STACKER_CROSSFIT = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_qwen_tfidf_crossfit.jsonl"
TFIDF_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_tfidf_validation.jsonl"
STACKED_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_stacked_validation.jsonl"

run_cli(
    "fit two-source stacker",
    RUN_TWO_SOURCE_STACKER,
    "scripts/fit_stacker.py",
    "--config", "configs/stacker.yaml",
    "--gold", TRAIN_PATH,
    "--folds", FOLDS_PATH,
    "--source", f"qwen={QWEN_OOF}",
    "--source", f"tfidf={TFIDF_OOF}",
    "--output", STACKER,
    "--crossfit-output", STACKER_CROSSFIT,
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_qwen_tfidf.json",
    "--model-name", f"{RUN_NAMESPACE}_qwen_tfidf",
)
run_cli(
    "validation TF-IDF",
    RUN_VALIDATION_TFIDF,
    "scripts/predict_baseline.py",
    "--input", VALIDATION_PATH,
    "--model", TFIDF_MODEL,
    "--output", TFIDF_VALIDATION,
)
run_cli(
    "apply validation stacker",
    RUN_APPLY_STACKER,
    "scripts/apply_stacker.py",
    "--stacker", STACKER,
    "--source", f"qwen={QWEN_VALIDATION_RAW}",
    "--source", f"tfidf={TFIDF_VALIDATION}",
    "--output", STACKED_VALIDATION,
)


### 7. 선택 후보: anchor/KNN
각 fold의 reference embedding은 같은 checkpoint 공간에서 만들며 held fold label을
anchor bank에서 제외한다. 아래 출력들은 Qwen+TF-IDF가 검증된 뒤에만 승격 후보로 쓴다.


In [ ]:
EMBEDDING_DIR = PROJECT_ROOT / "artifacts" / "embeddings" / RUN_NAMESPACE
ANCHOR_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_anchor_oof.jsonl"
ANCHOR_BANK = PROJECT_ROOT / "artifacts" / "anchors" / f"{RUN_NAMESPACE}_anchor_bank.npz"

for fold, checkpoint_path in enumerate(checkpoints):
    run_cli(
        f"reference embedding fold {fold}",
        RUN_ANCHOR_REFERENCE_EMBEDDINGS,
        "scripts/extract_embeddings.py",
        "--input", TRAIN_PATH,
        "--checkpoint", checkpoint_path,
        "--role", "reference",
        "--precision", "4bit",
        "--batch-size", "1",
        "--output", EMBEDDING_DIR / f"train_fold{fold}.npz",
        *download_flag,
    )

anchor_arguments: list[object] = [
    "scripts/build_anchor_oof.py", "--config", "configs/anchor_knn.yaml",
    "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
]
for fold in FOLDS:
    anchor_arguments.extend(["--embedding", f"{fold}={EMBEDDING_DIR / f'train_fold{fold}.npz'}"])
anchor_arguments.extend([
    "--oof-output", ANCHOR_OOF, "--anchor-bank", ANCHOR_BANK,
    "--diagnostics", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_anchor_neighbors.jsonl",
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_anchor.json",
    "--model-name", f"{RUN_NAMESPACE}_anchor",
])
run_cli("build anchor OOF", RUN_ANCHOR_OOF, *anchor_arguments)


### 8. 선택 후보: assessment-question Ridge

In [ ]:
ASSESSMENT_CACHE = PROJECT_ROOT / "artifacts" / "assessment" / f"{RUN_NAMESPACE}_train_features.npz"
ASSESSMENT_DIR = PROJECT_ROOT / "artifacts" / "assessment" / f"{RUN_NAMESPACE}_ridge"

run_cli(
    "cache assessment logits",
    RUN_ASSESSMENT_CACHE,
    "scripts/cache_assessment_logits.py",
    "--config", "configs/assessment_questions.yaml",
    "--input", TRAIN_PATH,
    "--model-revision", MODEL_REVISION,
    "--tokenizer-revision", MODEL_REVISION,
    "--output", ASSESSMENT_CACHE,
    *download_flag,
)
run_cli(
    "fit assessment branch",
    RUN_ASSESSMENT_FIT,
    "scripts/fit_assessment_branch.py",
    "--config", "configs/assessment_questions.yaml",
    "--gold", TRAIN_PATH,
    "--cache", ASSESSMENT_CACHE,
    "--folds", FOLDS_PATH,
    "--output-dir", ASSESSMENT_DIR,
    "--model-name", f"{RUN_NAMESPACE}_assessment",
)


### 9. auxiliary validation 예측
OOF에서 독립 gate를 통과한 후보만 validation에 적용한다. query embedding은 reference와
같은 fold checkpoint 공간에서 만들고, assessment는 train cache와 동일한 revision·질문
계약을 사용한다.


In [ ]:
TARGET_QUERY_DIR = PROJECT_ROOT / "artifacts" / "embeddings" / f"{RUN_NAMESPACE}_validation"
ANCHOR_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_anchor_validation.jsonl"
ASSESSMENT_VALIDATION_CACHE = (
    PROJECT_ROOT / "artifacts" / "assessment" / f"{RUN_NAMESPACE}_validation_features.npz"
)
ASSESSMENT_VALIDATION = (
    PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_assessment_validation.jsonl"
)

for fold, checkpoint_path in enumerate(checkpoints):
    run_cli(
        f"validation query embedding fold {fold}",
        RUN_TARGET_QUERY_EMBEDDINGS,
        "scripts/extract_embeddings.py",
        "--input", VALIDATION_PATH,
        "--checkpoint", checkpoint_path,
        "--role", "query",
        "--precision", "4bit",
        "--batch-size", "1",
        "--output", TARGET_QUERY_DIR / f"validation_fold{fold}.npz",
        *download_flag,
    )

target_anchor_arguments: list[object] = [
    "scripts/predict_anchor.py", "--input", VALIDATION_PATH, "--anchor-bank", ANCHOR_BANK,
]
for fold in FOLDS:
    target_anchor_arguments.extend([
        "--embedding", f"{fold}={TARGET_QUERY_DIR / f'validation_fold{fold}.npz'}",
    ])
target_anchor_arguments.extend([
    "--output", ANCHOR_VALIDATION,
    "--diagnostics", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_anchor_validation_neighbors.jsonl",
    "--model-name", f"{RUN_NAMESPACE}_anchor",
])
run_cli("predict validation anchor", RUN_TARGET_ANCHOR, *target_anchor_arguments)

run_cli(
    "cache validation assessment logits",
    RUN_TARGET_ASSESSMENT_CACHE,
    "scripts/cache_assessment_logits.py",
    "--config", "configs/assessment_questions.yaml",
    "--input", VALIDATION_PATH,
    "--model-revision", MODEL_REVISION,
    "--tokenizer-revision", MODEL_REVISION,
    "--output", ASSESSMENT_VALIDATION_CACHE,
    *download_flag,
)
run_cli(
    "predict validation assessment branch",
    RUN_TARGET_ASSESSMENT_PREDICTION,
    "scripts/predict_assessment_branch.py",
    "--input", VALIDATION_PATH,
    "--cache", ASSESSMENT_VALIDATION_CACHE,
    "--model", ASSESSMENT_DIR / "assessment_ridge.json",
    "--output", ASSESSMENT_VALIDATION,
)


### 10. 네 source 후보 stacker

In [ ]:
MULTISOURCE_STACKER = PROJECT_ROOT / "artifacts" / "stackers" / f"{RUN_NAMESPACE}_multisource.json"
MULTISOURCE_CROSSFIT = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_crossfit.jsonl"
MULTISOURCE_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_validation.jsonl"
ASSESSMENT_OOF = ASSESSMENT_DIR / "assessment_oof.jsonl"

valid_auxiliary_sources = {"anchor", "assessment"}
unknown_sources = set(PROMOTED_AUXILIARY_SOURCES).difference(valid_auxiliary_sources)
if unknown_sources:
    raise ValueError(f"알 수 없는 auxiliary source: {sorted(unknown_sources)}")
if (RUN_MULTISOURCE_STACKER or RUN_APPLY_MULTISOURCE) and not PROMOTED_AUXILIARY_SOURCES:
    raise ValueError("auxiliary source가 없으면 앞 절의 two-source stacker를 사용하세요.")

fit_multisource_arguments: list[object] = [
    "scripts/fit_stacker.py",
    "--config", "configs/stacker.yaml",
    "--gold", TRAIN_PATH,
    "--folds", FOLDS_PATH,
    "--source", f"qwen={QWEN_OOF}",
    "--source", f"tfidf={TFIDF_OOF}",
]
oof_auxiliary_paths = {"anchor": ANCHOR_OOF, "assessment": ASSESSMENT_OOF}
for alias in PROMOTED_AUXILIARY_SOURCES:
    fit_multisource_arguments.extend(["--source", f"{alias}={oof_auxiliary_paths[alias]}"])
fit_multisource_arguments.extend([
    "--output", MULTISOURCE_STACKER,
    "--crossfit-output", MULTISOURCE_CROSSFIT,
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_multisource.json",
    "--model-name", f"{RUN_NAMESPACE}_multisource",
])
run_cli("fit promoted multisource stacker", RUN_MULTISOURCE_STACKER, *fit_multisource_arguments)

apply_multisource_arguments: list[object] = [
    "scripts/apply_stacker.py",
    "--stacker", MULTISOURCE_STACKER,
    "--source", f"qwen={QWEN_VALIDATION_RAW}",
    "--source", f"tfidf={TFIDF_VALIDATION}",
]
target_auxiliary_paths = {"anchor": ANCHOR_VALIDATION, "assessment": ASSESSMENT_VALIDATION}
for alias in PROMOTED_AUXILIARY_SOURCES:
    apply_multisource_arguments.extend(["--source", f"{alias}={target_auxiliary_paths[alias]}"])
apply_multisource_arguments.extend(["--output", MULTISOURCE_VALIDATION])
run_cli("apply validation multisource stacker", RUN_APPLY_MULTISOURCE, *apply_multisource_arguments)


### 11. 승격과 다음 단계
stacker에는 calibrated Qwen이 아니라 동일 raw Qwen signature를 넣는다. anchor와
assessment는 각 OOF gate 및 시간 예산을 통과한 경우에만 같은 alias로 fit/apply 양쪽에
추가한다. 확정된 genuine cross-fit score는 `05_rationale_and_finalization.ipynb`로 넘긴다.


---

<a id="part-05"></a>

# PART 6 · Rationale·최종화

**원본 노트북:** `05_rationale_and_finalization.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 05. Grounded Rationale 학습 및 최종화
최종 score를 먼저 고정한 뒤 genuine OOF score로 silver를 만들고 rationale adapter를
학습한다. score는 rationale 단계에서 절대 수정하지 않으며, adapter가 승격되지 않으면
deterministic exact-evidence fallback으로 최종 스키마를 만든다.


### 2. 파라미터와 실행 스위치

In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"

# 04에서 실제 승격된 하나의 score branch를 선택한다.
SELECTED_SCORE_BRANCH = "two_source"  # "two_source" 또는 "multisource"
score_branches = {
    "two_source": {
        "model": f"{RUN_NAMESPACE}_qwen_tfidf",
        "oof": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_qwen_tfidf_crossfit.jsonl",
        "target": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_stacked_validation.jsonl",
    },
    "multisource": {
        "model": f"{RUN_NAMESPACE}_multisource",
        "oof": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_crossfit.jsonl",
        "target": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_validation.jsonl",
    },
}
if SELECTED_SCORE_BRANCH not in score_branches:
    raise ValueError(f"지원하지 않는 score branch: {SELECTED_SCORE_BRANCH}")
selected_scores = score_branches[SELECTED_SCORE_BRANCH]
SCORE_MODEL = selected_scores["model"]
OOF_SCORES = selected_scores["oof"]
TARGET_SCORES = selected_scores["target"]

RUN_SILVER = False
RUN_RATIONALE_TRAINING = False
RUN_ADAPTER_GENERATION = False
RUN_ADAPTER_FINALIZATION = False
RUN_FALLBACK_FINALIZATION = False
RUN_FINAL_SCHEMA_EVALUATION = False
RUN_BLIND_REVIEW_PACK = False
RUN_LOCAL_GGUF_JUDGE = False
RUN_JUDGE_SUMMARY = False
INSTALL_LOCAL_JUDGE_DEPENDENCY = False
FINAL_SCHEMA_VARIANT = "fallback"  # "fallback" 또는 "adapter"
if FINAL_SCHEMA_VARIANT not in {"fallback", "adapter"}:
    raise ValueError(f"지원하지 않는 final schema variant: {FINAL_SCHEMA_VARIANT}")

GPU_ENABLED = RUN_SILVER or RUN_RATIONALE_TRAINING or RUN_ADAPTER_GENERATION
require_model_revision(MODEL_REVISION, enabled=GPU_ENABLED)
require_bf16_gpu(enabled=GPU_ENABLED, recommended_vram_gb=35)


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


### 3. genuine OOF 기반 grounded silver

In [ ]:
SILVER = PROJECT_ROOT / "artifacts" / "rationale" / f"{RUN_NAMESPACE}_silver_accepted.jsonl"
SILVER_REJECTED = PROJECT_ROOT / "artifacts" / "rationale" / f"{RUN_NAMESPACE}_silver_rejected.jsonl"
SILVER_REPORT = PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_rationale_silver.json"

run_cli(
    "build grounded silver",
    RUN_SILVER,
    "scripts/build_silver_rationales.py",
    "--config", "configs/rationale_sft.yaml",
    "--input", TRAIN_PATH,
    "--scores", OOF_SCORES,
    "--folds", FOLDS_PATH,
    "--model-revision", MODEL_REVISION,
    "--output", SILVER,
    "--rejected-output", SILVER_REJECTED,
    "--report", SILVER_REPORT,
    *download_flag,
)
silver_report = show_json(SILVER_REPORT)
if RUN_RATIONALE_TRAINING:
    if not silver_report or not silver_report.get("promotion_eligible", False):
        raise RuntimeError("silver acceptance gate를 통과하지 못했습니다.")


### 4. rationale QLoRA SFT

In [ ]:
RATIONALE_RUN_ID = f"rationale_{RUN_NAMESPACE}"
RATIONALE_MODEL_DIR = PROJECT_ROOT / "artifacts" / "models" / RATIONALE_RUN_ID

run_cli(
    "train rationale adapter",
    RUN_RATIONALE_TRAINING,
    "-m", "src.train.train_rationale",
    "--config", "configs/rationale_sft.yaml",
    "--input", TRAIN_PATH,
    "--silver", SILVER,
    "--run-id", RATIONALE_RUN_ID,
    "--model-revision", MODEL_REVISION,
    *download_flag,
)


### 5. adapter rationale 생성과 최종 스키마

In [ ]:
TARGET_INPUT = VALIDATION_PATH  # 최종 test에서는 TEST_PATH로 바꾼다.
RATIONALE_OUTPUT = PROJECT_ROOT / "artifacts" / "rationale" / f"{RUN_NAMESPACE}_target_grounded.jsonl"
RATIONALE_REPORT = PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_target_rationale.json"
ADAPTER_FINAL = PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_adapter_validation.jsonl"
FALLBACK_FINAL = PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_fallback_validation.jsonl"

run_cli(
    "generate adapter rationales",
    RUN_ADAPTER_GENERATION,
    "scripts/generate_rationales.py",
    "--input", TARGET_INPUT,
    "--scores", TARGET_SCORES,
    "--checkpoint", RATIONALE_MODEL_DIR / "best.json",
    "--precision", "4bit",
    "--max-attempts", "2",
    "--output", RATIONALE_OUTPUT,
    "--report", RATIONALE_REPORT,
    *download_flag,
)
run_cli(
    "adapter finalization",
    RUN_ADAPTER_FINALIZATION,
    "scripts/finalize_predictions.py",
    "--input", TARGET_INPUT,
    "--scores", TARGET_SCORES,
    "--rationales", RATIONALE_OUTPUT,
    "--output", ADAPTER_FINAL,
    "--ledger", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_adapter_ledger.jsonl",
    "--bare-output", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_adapter_submission.jsonl",
    "--model-name", f"{SCORE_MODEL}_adapter_rationale",
)
run_cli(
    "deterministic fallback finalization",
    RUN_FALLBACK_FINALIZATION,
    "scripts/finalize_predictions.py",
    "--input", TARGET_INPUT,
    "--scores", TARGET_SCORES,
    "--output", FALLBACK_FINAL,
    "--ledger", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_fallback_ledger.jsonl",
    "--bare-output", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_fallback_submission.jsonl",
    "--model-name", f"{SCORE_MODEL}_deterministic_fallback",
)


### 6. final schema 평가 및 blind review

In [ ]:
run_cli(
    "final schema evaluation",
    RUN_FINAL_SCHEMA_EVALUATION,
    "scripts/evaluate_predictions.py",
    "--gold", VALIDATION_PATH,
    "--pred", {"fallback": FALLBACK_FINAL, "adapter": ADAPTER_FINAL}[FINAL_SCHEMA_VARIANT],
    "--final-schema",
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_final_schema.json",
)

REVIEW_PACK = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_blind_100.jsonl"
REVIEW_KEY = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_blind_100_key.jsonl"
run_cli(
    "blind review pack",
    RUN_BLIND_REVIEW_PACK,
    "scripts/build_rationale_review_pack.py",
    "--input", VALIDATION_PATH,
    "--candidate", ADAPTER_FINAL,
    "--baseline", FALLBACK_FINAL,
    "--sample-size", "100",
    "--seed", "2026",
    "--output", REVIEW_PACK,
    "--key-output", REVIEW_KEY,
)


### 7. 선택적 로컬 GGUF judge

In [ ]:
GGUF_MODEL = Path("/content/drive/MyDrive/models/pinned-rationale-judge.Q4_K_M.gguf")
JUDGMENTS = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_judgments.jsonl"
JUDGE_SUMMARY = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_judge_summary.json"

if INSTALL_LOCAL_JUDGE_DEPENDENCY:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[judge]"],
        cwd=PROJECT_ROOT,
    )

run_cli(
    "local GGUF judge",
    RUN_LOCAL_GGUF_JUDGE,
    "scripts/judge_rationales_local.py",
    "--config", "configs/rationale_judge.yaml",
    "--review-pack", REVIEW_PACK,
    "--model", GGUF_MODEL,
    "--output", JUDGMENTS,
)
run_cli(
    "summarize judge after judging",
    RUN_JUDGE_SUMMARY,
    "scripts/summarize_rationale_judge.py",
    "--judgments", JUDGMENTS,
    "--key", REVIEW_KEY,
    "--output", JUDGE_SUMMARY,
)


### 8. 승격 규칙
adapter와 fallback은 동일한 score 파일을 사용해야 한다. 사람 blind 평가에서 adapter가
우세할 때만 승격하며 자동 judge는 보조 신호일 뿐이다. 어느 경로든 ledger는 내부 감사용이고
bare submission에는 포함하지 않는다.


---

<a id="part-06"></a>

# PART 7 · 제출·테스트

**원본 노트북:** `06_submission_and_tests.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 06. 제출 추론, 패키징 및 테스트
외부에서 제공된 unlabeled `test.jsonl`을 검증하고 승격된 artifact로 추론·최종화를
수행한다. 실제 제출 패키징과 L40S smoke는 필요한 lock/license/L40S가 모두 준비된
경우에만 실행한다. 일반 Colab GPU 결과를 공식 L40S 증거로 간주하지 않는다.


### 2. 외부 입력과 실행 스위치

In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"
TEMPLATE_CONFIG = PROJECT_ROOT / "configs" / "inference_l40s.yaml"
# scorer artifact namespace와 runtime config namespace를 분리한다.
SCORER_RUN_NAMESPACE = RUN_NAMESPACE
RUNTIME_CONFIG_TAG = f"{RUN_NAMESPACE}_colab"  # 공식 package는 예: "v1_l40s"
INFERENCE_CONFIG = PROJECT_ROOT / "artifacts" / "runtime_configs" / f"inference_{RUNTIME_CONFIG_TAG}.yaml"
REQUIREMENTS_LOCK = PROJECT_ROOT / "requirements-lock.txt"
LICENSE_FILES = [PROJECT_ROOT / "LICENSE", PROJECT_ROOT / "THIRD_PARTY_NOTICES.md"]
HF_HOME_PATH = Path(os.environ.get("HF_HOME", "")) if os.environ.get("HF_HOME") else None

SCORER_EXPERIMENT_ID = f"qwen3_scorer_{SCORER_RUN_NAMESPACE}"
SCORER_SEED = 42
FIXED_EPOCH = 3
SCORER_NAME = f"{SCORER_EXPERIMENT_ID}_seed{SCORER_SEED}"
SCORER_CHECKPOINTS = [
    PROJECT_ROOT / "artifacts" / "models" / SCORER_EXPERIMENT_ID
    / f"seed_{SCORER_SEED}__fold_{fold}" / f"epoch_{FIXED_EPOCH}"
    for fold in range(5)
]
SCORING_MODE = "qwen_ensemble"  # 또는 "qwen_multisource_stacker"
QWEN_CALIBRATOR = PROJECT_ROOT / "artifacts" / "calibrators" / f"{SCORER_EXPERIMENT_ID}_affine.json"
BASELINE_ARTIFACT = None
ANCHOR_ARTIFACT = None
ASSESSMENT_ARTIFACT = None
STACKER_ARTIFACT = None
STACKER_SOURCE_ALIASES = {
    "qwen": "qwen",
    "baseline": None,
    "anchor": None,
    "assessment": None,
}
RATIONALE_MODE = "deterministic"  # 또는 "adapter"
RATIONALE_CHECKPOINT = None
EXPECTED_TEST_ROWS = 400
# Colab 기능 검증은 None, 공식 package/L40S 검증은 "L40S"로 새 config를 만든다.
REQUIRED_GPU_NAME_CONTAINS = None

RUN_STATIC_TESTS = False
RUN_TEST_INPUT_CHECK = False
BUILD_RUNTIME_CONFIG = False
RUN_SUBMISSION_ENGINE = False
RUN_PACKAGE = False
RUN_L40S_SMOKE = False

require_model_revision(
    MODEL_REVISION,
    enabled=BUILD_RUNTIME_CONFIG or RUN_SUBMISSION_ENGINE or RUN_PACKAGE or RUN_L40S_SMOKE,
)
require_bf16_gpu(enabled=RUN_SUBMISSION_ENGINE or RUN_L40S_SMOKE)


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


### 3. 정적 회귀 테스트

In [ ]:
run_cli("full unit tests", RUN_STATIC_TESTS, "-m", "pytest", "-q")


### 4. test JSONL 계약 검사

In [ ]:
if RUN_TEST_INPUT_CHECK:
    require_paths(TEST_PATH)
    from src.data.load import load_inference_jsonl

    records = load_inference_jsonl(TEST_PATH)
    ids = [record.id for record in records]
    if len(ids) != len(set(ids)):
        raise ValueError("test id가 중복되었습니다.")
    print({"rows": len(records), "first_id": ids[0] if ids else None})
else:
    print("공식 test 파일은 저장소에 포함되어 있지 않습니다:", TEST_PATH)


### 5. fail-closed runtime config 생성
소스 template를 직접 수정하지 않고, 승격된 checkpoint/revision/source를 주입한 새 YAML을
`artifacts/runtime_configs/`에 create-only로 만든다. multisource 모드에서는 raw Qwen을
사용하므로 `QWEN_CALIBRATOR=None`이어야 하며, artifact와 alias를 쌍으로 채운다.
A100/L4 Colab 기능 검증은 `REQUIRED_GPU_NAME_CONTAINS=None`, 공식 패키지는 `"L40S"`를 쓴다.


In [ ]:
if BUILD_RUNTIME_CONFIG:
    import yaml

    require_paths(TEMPLATE_CONFIG, *SCORER_CHECKPOINTS)
    if HF_HOME_PATH is None or not HF_HOME_PATH.is_dir():
        raise FileNotFoundError("offline 실행에 사용할 고정 HF_HOME을 먼저 설정하세요.")
    if INFERENCE_CONFIG.exists():
        raise FileExistsError(f"runtime config는 덮어쓰지 않습니다: {INFERENCE_CONFIG}")
    if SCORING_MODE not in {"qwen_ensemble", "qwen_multisource_stacker"}:
        raise ValueError(f"지원하지 않는 SCORING_MODE: {SCORING_MODE}")
    if RATIONALE_MODE not in {"deterministic", "adapter"}:
        raise ValueError(f"지원하지 않는 RATIONALE_MODE: {RATIONALE_MODE}")
    if SCORING_MODE == "qwen_multisource_stacker" and QWEN_CALIBRATOR is not None:
        raise ValueError("raw Qwen stacker 모드에서는 QWEN_CALIBRATOR=None이어야 합니다.")
    if RATIONALE_MODE == "adapter" and RATIONALE_CHECKPOINT is None:
        raise ValueError("adapter rationale에는 RATIONALE_CHECKPOINT가 필요합니다.")

    optional_artifacts = {
        "baseline": BASELINE_ARTIFACT,
        "anchor": ANCHOR_ARTIFACT,
        "assessment": ASSESSMENT_ARTIFACT,
    }
    if SCORING_MODE == "qwen_ensemble":
        if any(value is not None for value in optional_artifacts.values()) or STACKER_ARTIFACT is not None:
            raise ValueError("qwen_ensemble에서는 auxiliary/stacker artifact를 모두 None으로 둡니다.")
    else:
        if STACKER_ARTIFACT is None or not any(value is not None for value in optional_artifacts.values()):
            raise ValueError("multisource에는 stacker와 하나 이상의 auxiliary artifact가 필요합니다.")
        for kind, artifact in optional_artifacts.items():
            if (artifact is None) != (STACKER_SOURCE_ALIASES[kind] is None):
                raise ValueError(f"{kind} artifact와 alias를 함께 설정하세요.")

    config = yaml.safe_load(TEMPLATE_CONFIG.read_text(encoding="utf-8"))
    config["project_root"] = str(PROJECT_ROOT)
    config["runtime"].update({
        "hf_home": str(HF_HOME_PATH.resolve()),
        "package_manifest": None,
        "requirements_lock": str(REQUIREMENTS_LOCK.resolve()) if REQUIREMENTS_LOCK.exists() else None,
        "required_gpu_name_contains": REQUIRED_GPU_NAME_CONTAINS,
        "expected_rows": EXPECTED_TEST_ROWS,
    })
    config["scoring"]["mode"] = SCORING_MODE
    config["scoring"]["qwen"].update({
        "scorer_name": SCORER_NAME,
        "checkpoints": [str(path.resolve()) for path in SCORER_CHECKPOINTS],
        "model_revision": MODEL_REVISION,
        "calibrator": str(QWEN_CALIBRATOR.resolve()) if QWEN_CALIBRATOR is not None else None,
    })
    for kind, artifact in optional_artifacts.items():
        config["scoring"][kind]["artifact"] = (
            str(Path(artifact).resolve()) if artifact is not None else None
        )
    config["scoring"]["stacker"]["artifact"] = (
        str(Path(STACKER_ARTIFACT).resolve()) if STACKER_ARTIFACT is not None else None
    )
    config["scoring"]["stacker"]["source_aliases"] = dict(STACKER_SOURCE_ALIASES)
    config["rationale"]["mode"] = RATIONALE_MODE
    config["rationale"]["checkpoint"] = (
        str(Path(RATIONALE_CHECKPOINT).resolve()) if RATIONALE_CHECKPOINT is not None else None
    )

    referenced = [
        *(path for path in optional_artifacts.values() if path is not None),
        *(path for path in (QWEN_CALIBRATOR, STACKER_ARTIFACT, RATIONALE_CHECKPOINT) if path is not None),
    ]
    require_paths(*[Path(path) for path in referenced])
    INFERENCE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    INFERENCE_CONFIG.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("created", INFERENCE_CONFIG)
else:
    print("runtime config:", INFERENCE_CONFIG, "ready" if INFERENCE_CONFIG.exists() else "not created")


### 6. offline 제출 엔진
`configs/inference_l40s.yaml`의 모든 `PLACEHOLDER`와 null revision을 승격된 artifact로
먼저 채운다. raw Qwen으로 학습한 stacker라면 Qwen calibrator는 null이어야 한다.


In [ ]:
FINAL_OUTPUT = PROJECT_ROOT / "artifacts" / "final" / f"{RUNTIME_CONFIG_TAG}_run_submission.jsonl"
FINAL_LEDGER = PROJECT_ROOT / "artifacts" / "final" / f"{RUNTIME_CONFIG_TAG}_run_submission_ledger.jsonl"
BARE_OUTPUT = PROJECT_ROOT / "artifacts" / "final" / f"{RUNTIME_CONFIG_TAG}_submission.jsonl"

run_cli(
    "submission engine",
    RUN_SUBMISSION_ENGINE,
    "scripts/run_submission.py",
    "--config", INFERENCE_CONFIG,
    "--input", TEST_PATH,
    "--output", FINAL_OUTPUT,
    "--ledger", FINAL_LEDGER,
    "--bare-output", BARE_OUTPUT,
)


### 7. 제출 패키징

In [ ]:
PACKAGE_DIR = PROJECT_ROOT / "artifacts" / "packages" / f"submission_{RUNTIME_CONFIG_TAG}"
if RUN_PACKAGE:
    import yaml

    require_paths(REQUIREMENTS_LOCK, *LICENSE_FILES)
    if REQUIRED_GPU_NAME_CONTAINS != "L40S":
        raise ValueError("공식 package config는 REQUIRED_GPU_NAME_CONTAINS='L40S'여야 합니다.")
    packaged_source_config = yaml.safe_load(INFERENCE_CONFIG.read_text(encoding="utf-8"))
    if packaged_source_config["runtime"]["required_gpu_name_contains"] != "L40S":
        raise ValueError("기존 runtime config가 L40S 전용이 아닙니다. 새 RUN_NAMESPACE로 다시 만드세요.")
    if HF_HOME_PATH is None or not HF_HOME_PATH.exists():
        raise FileNotFoundError("고정된 전용 HF_HOME cache가 필요합니다.")

package_arguments: list[object] = [
    "scripts/package_submission.py",
    "--config", INFERENCE_CONFIG,
    "--output-dir", PACKAGE_DIR,
    "--requirements-lock", REQUIREMENTS_LOCK,
]
for path in LICENSE_FILES:
    package_arguments.extend(["--license-file", path])
if HF_HOME_PATH is not None:
    package_arguments.extend(["--hf-home", HF_HOME_PATH])
run_cli("package submission", RUN_PACKAGE, *package_arguments)


### 8. 실제 L40S offline smoke

In [ ]:
if RUN_L40S_SMOKE:
    import torch

    gpu_name = torch.cuda.get_device_name(0)
    if "L40S" not in gpu_name:
        raise RuntimeError(f"공식 smoke는 L40S에서만 실행합니다. 현재 GPU: {gpu_name}")

run_cli(
    "L40S offline smoke",
    RUN_L40S_SMOKE,
    "scripts/smoke_test_submission.py",
    "--config", PACKAGE_DIR / "inference_l40s.yaml",
    "--input", TEST_PATH,
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"submission_{RUNTIME_CONFIG_TAG}_l40s_smoke.json",
    "--expected-count", "400",
    "--offline",
    "--strict",
    "--verify-determinism",
    "--require-package-manifest",
)


### 9. 완료 조건
- strict parse 100%, ID/순서 보존, score byte 동일성
- package manifest와 모든 payload hash 일치
- 정확히 고정된 dependency lock 및 license/notice 포함
- 네트워크 차단 L40S 400건 시간·메모리·결정론 보고서 생성


---

<a id="part-90"></a>

# PART 8 · 선택적 최종 재훈련

**원본 노트북:** `90_optional_final_retraining.ipynb`  
공통 bootstrap과 dependency 설치는 맨 앞에서 통합했으며, 이 PART에는 원본의
설정·실행·검증 셀을 순서대로 포함한다.


## 90. 규정 승인 후 선택적 train+validation 최종 재훈련
**일반 개발 노트북이 아니다.** 운영 규정 또는 운영진 답변이 validation label의 최종
학습 사용을 명시적으로 허용한 경우에만 실행한다. 불명확하면 이 노트북을 닫고
2,000건 train artifact를 유지한다.

모든 출력은 `artifacts/final_train_validation/` 아래 새 namespace로 만들며 기존 OOF,
calibrator, stacker와 절대 혼합하지 않는다.


### 2. 이중 승인 gate

In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

RULES_ALLOW_VALIDATION_LABEL_TRAINING = False
ACKNOWLEDGEMENT = ""
REQUIRED_ACKNOWLEDGEMENT = "규정이 validation label의 최종 학습 사용을 명시적으로 허용함을 확인했습니다"

RUN_COMBINED_DATASET = False
RUN_FINAL_FOLDS = False
RUN_FINAL_POLICY = False
PLAN_FINAL_TRAINING = False
EXECUTE_FINAL_TRAINING = False

FINAL_EXPERIMENT_ID = f"qwen3_scorer_final_2400_{RUN_NAMESPACE}"
FINAL_ROOT = PROJECT_ROOT / "artifacts" / "final_train_validation"
FINAL_FOLDS = FINAL_ROOT / "folds" / "folds_5fold_seed42.jsonl"
FINAL_POLICY = FINAL_ROOT / "policies" / "scorer_epoch3.json"

ANY_FINAL_ACTION = any([
    RUN_COMBINED_DATASET, RUN_FINAL_FOLDS, RUN_FINAL_POLICY,
    PLAN_FINAL_TRAINING, EXECUTE_FINAL_TRAINING,
])
if ANY_FINAL_ACTION:
    if not RULES_ALLOW_VALIDATION_LABEL_TRAINING:
        raise PermissionError("규정 승인 boolean이 False입니다.")
    if ACKNOWLEDGEMENT != REQUIRED_ACKNOWLEDGEMENT:
        raise PermissionError("승인 문구가 정확히 일치하지 않습니다.")
require_model_revision(MODEL_REVISION, enabled=PLAN_FINAL_TRAINING or EXECUTE_FINAL_TRAINING)
require_bf16_gpu(enabled=EXECUTE_FINAL_TRAINING)


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


### 3. immutable 2,400건 결합 artifact

In [ ]:
run_cli(
    "build final combined dataset",
    RUN_COMBINED_DATASET,
    "scripts/build_final_combined_data.py",
    "--config", "configs/data_final_combined.yaml",
    "--acknowledge-rules-allow-validation-label-training",
)


### 4. 전용 fold와 고정 epoch 정책

In [ ]:
run_cli(
    "final-only folds",
    RUN_FINAL_FOLDS,
    "scripts/make_folds.py",
    "--config", "configs/data_final_combined.yaml",
    "--n-folds", "5",
    "--seed", "42",
    "--output", FINAL_FOLDS,
)
run_cli(
    "final-only epoch policy",
    RUN_FINAL_POLICY,
    "scripts/select_fixed_epoch.py",
    "--preselected-epoch", "3",
    "--reason", "2,000건 개발 단계에서 잠근 최종 epoch 정책",
    "--output", FINAL_POLICY,
)


### 5. 전용 fold×seed plan 및 실행

In [ ]:
def final_training_arguments(*, execute: bool) -> list[object]:
    arguments: list[object] = [
        "scripts/orchestrate_scorer_training.py",
        "--config", "configs/scorer_qlora.yaml",
        "--data-config", "configs/data_final_combined.yaml",
        "--experiment-id", FINAL_EXPERIMENT_ID,
        "--folds-file", FINAL_FOLDS,
        "--epoch-policy", FINAL_POLICY,
        "--model-revision", MODEL_REVISION,
        "--seed", "42", "--seed", "1337", "--seed", "2026",
    ]
    if execute:
        arguments.append("--execute")
    arguments.extend(download_flag)
    return arguments

run_cli("final training plan", PLAN_FINAL_TRAINING, *final_training_arguments(execute=False))
run_cli("execute final training", EXECUTE_FINAL_TRAINING, *final_training_arguments(execute=True))


### 6. 이후 단계
새 fold×seed checkpoint로 OOF, calibrator, auxiliary source, stacker를 처음부터 다시 만든다.
2,000건 개발 artifact의 manifest나 scorer signature를 재사용하지 않는다. 이 경로의 성능
승격과 제출은 별도의 검증 결과가 있을 때만 허용한다.


---

# END · 실행 결과 확인

각 PART가 생성한 artifact와 manifest는 동일한 Google Drive `artifacts/` 경로에 남는다.
세션을 다시 시작할 때는 이 마스터 노트북의 공통 런타임을 먼저 실행하고, 동일한
namespace·model revision·Drive 경로를 유지한 채 필요한 PART부터 계속한다.
